In [ ]:
import pandas as pd
import os

In [ ]:
cwd= os.getcwd()
print(cwd)
BASE_DIR= os.path.join(cwd,"..")

data= os.path.join(BASE_DIR, "data", "concat_for_eda.csv")

In [ ]:
df= pd.read_csv(data)
# df= pd.read_excel(data)
print(df)
print(df.shape)

In [ ]:
df.info()

### Descriptive Statistics by Diagnosis Group

In [ ]:
labels = ["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]
exclude = labels + ["Giới_nữ"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

for label in labels:
    subset = df[df[label] == 1][numeric_cols]
    n = len(subset)
    print(f"{label} (n={n})")
    print(subset.describe().round(2).to_string())

### Missing Value Rate by Diagnosis Group
Heatmap showing the percentage of missing values for each feature within each diagnosis group. Supports the argument for not imputing — each diagnosis group has a different missing data pattern.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

labels = ["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]
exclude = labels + ["Giới_nữ"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

missing_by_diag = pd.DataFrame()
for label in labels:
    subset = df[df[label] == 1][numeric_cols]
    missing_pct = (subset.isnull().sum() / len(subset) * 100).round(1)
    missing_by_diag[label] = missing_pct

missing_by_diag = missing_by_diag[missing_by_diag.max(axis=1) > 0]

plt.figure(figsize=(10, 8))
sns.heatmap(missing_by_diag, annot=True, fmt=".1f", cmap="YlOrRd", linewidths=0.5)
plt.title("Missing Value Rate (%) by Diagnosis Group")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("missing_by_diagnosis.png", dpi=300)
plt.show()

### Kruskal-Wallis Test
Compare the distribution of each feature across the 4 diagnosis groups (ACD, IDA, Alpha thalassemia, Beta thalassemia).

In [ ]:
from scipy.stats import kruskal
import warnings

labels = ["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]
exclude = labels + ["Giới_nữ"]
numeric_cols = df.select_dtypes(include='number').columns.difference(exclude).tolist()

kw_results = []

for col in numeric_cols:
    groups = []
    for label in labels:
        vals = df[df[label] == 1][col].dropna()
        if len(vals) >= 3:
            groups.append(vals)

    if len(groups) >= 2:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore", RuntimeWarning)
            stat, p = kruskal(*groups)

        if not pd.isna(stat):
            kw_results.append({
                "Feature": col,
                "H-statistic": round(stat, 4),
                "p-value": round(p, 6),
                "Significant": "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else ""
            })

kw_df = pd.DataFrame(kw_results).sort_values("p-value")
print(kw_df.to_string(index=False))

sig_features = kw_df[kw_df["Significant"] != ""]["Feature"].tolist()
print(f"\nStatistically significant features (p<0.05): {sig_features}")

In [ ]:
# add Dunn's test for post-hoc analysis
# Kruskal-Wallis + Dunn's test:
# Trả lời được cả 2 câu: "có khác nhau không" và "nhóm nào vs nhóm nào"

### Univariable Distribution

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df_copy= df.copy()

X_y_don= df_copy.drop(columns= labels)


In [ ]:
null_X = X_y_don.drop(columns= ["Chẩn đoán"])
null_counts = null_X.isnull().sum()

# Vẽ biểu đồ
plt.figure(figsize=(10, 5))
null_counts.plot(kind='bar')

# Xoay trục x 45 độ
plt.xticks(rotation=80)

plt.title('Number of Missing Values per Column')
plt.xlabel('Columns')
plt.ylabel('Count of Nulls')

plt.tight_layout()
plt.show()

In [ ]:
for i in range(len(X_y_don)):
    col_name= X_y_don.columns[i]
    plt.figure()
    sns.boxplot(data= X_y_don, x=col_name, hue= "Chẩn đoán")
    plt.legend(
    title= f"Distribution of {col_name} by diagnosis",
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)
    plt.show()

In [ ]:
plt.figure()
sns.countplot(data= X_y_don, x= "Giới_nữ", hue= "Chẩn đoán")
plt.legend(
    title= "Distribution of gender by diagnosis",
    bbox_to_anchor=(1.05, 1),
    loc='upper left'
)

In [ ]:
# làm cái này với điều kiện no null values

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = X_y_don.drop("Chẩn đoán", axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

sns.scatterplot(x=X_pca[:,0], y=X_pca[:,1], hue=X_y_don["Chẩn đoán"])

In [ ]:
from sklearn.manifold import TSNE
# làm cái này với điều kiện no null values

X = df.drop("Chẩn đoán", axis=1)
X_embedded = TSNE(n_components=2).fit_transform(X)

plt.figure()
sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=df["Chẩn đoán"])
plt.legend(
    title="Chẩn đoán",
    bbox_to_anchor=(1.05, 1),  # đẩy sang phải
    loc='upper left'
)
plt.show()

In [ ]:
from sklearn.metrics import silhouette_score

# làm cái này với điều kiện no null values

X = df.drop("Chẩn đoán", axis=1)
X_embedded = TSNE(n_components=2).fit_transform(X)
score = silhouette_score(X_embedded, df["Chẩn đoán"])
print(score)

In [ ]:
plt.figure()
# làm cái này với điều kiện no null values

X = df.drop("Chẩn đoán", axis=1)
X_embedded = TSNE(n_components=2).fit_transform(X)

ax = sns.kdeplot(
    x=X_embedded[:,0],
    y=X_embedded[:,1],
    hue=df["Chẩn đoán"]
)

sns.move_legend(ax, "upper left", bbox_to_anchor=(1, 1))

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
# làm cái này với điều kiện no null values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=3)
clusters = kmeans.fit_predict(X_scaled)

sns.scatterplot(x=X_embedded[:,0], y=X_embedded[:,1], hue=clusters)

In [ ]:
sns.histplot(df["Chẩn đoán"])
plt.xticks(rotation=90)
plt.show()

In [ ]:
y = df[["ACD", "IDA", "Alpha thalassemia" ,"Beta thalassemia"]]

counts = y.value_counts()

print(counts)

In [ ]:
import pandas as pd

# Các nhãn
labels = ["ACD", "IDA", "Alpha thalassemia", "Beta thalassemia"]

# Ma trận đồng xuất hiện
co_matrix = pd.DataFrame(
    0,
    index=labels,
    columns=labels
)

# Đếm số lần xuất hiện cùng nhau
for _, row in y.iterrows():

    active_labels = row[row == 1].index.tolist()

    for l1 in active_labels:
        for l2 in active_labels:
            co_matrix.loc[l1, l2] += 1

print(co_matrix)